In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from safetensors.torch import load_file

# Configuration
RESULTS_DIR = Path("../results/model_comparison/similarities")
PLOTS_DIR = Path("../plots/models_similarity")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Identifiers must match what was used in the script (split by /)
MODEL_A_ID = "llava-1.5-7b-hf"  # Finetuned (Rows)
MODEL_B_ID = "llava-v1.5-7b-pretrained-lbasile" # Pretrained (Columns)

DATASETS = {
    "cocoqa_txt": "Text QA",
    "cocoqa_img": "Image QA"
}

MEASURES = {
    "neighborhood_overlap": "Neighborhood Overlap",
    "linear_cka": "Linear CKA",
    "distance_correlation": "Distance Correlation"
}

In [ ]:
def load_data(dataset_key, measure_key):
    # Filename format: {dataset}_{model_a}_vs_{model_b}_{measure}.safetensors
    filename = f"{dataset_key}_{MODEL_A_ID}_vs_{MODEL_B_ID}_{measure_key}.safetensors"
    path = RESULTS_DIR / filename
    try:
        data = load_file(str(path))
        tensor = data[measure_key]
        return tensor.cpu().numpy()
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return None

def plot_model_comparison(measure_key, measure_title):
    # Load data for both datasets
    data_txt = load_data("cocoqa_txt", measure_key)
    data_img = load_data("cocoqa_img", measure_key)
    
    if data_txt is None and data_img is None:
        print(f"Skipping plot for {measure_title} due to missing data.")
        return

    # Setup figure
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
    
    datasets_list = [
        ("cocoqa_txt", "Text QA", data_txt), 
        ("cocoqa_img", "Image QA", data_img)
    ]
    
    for idx, (ds_key, ds_title, data) in enumerate(datasets_list):
        ax = axes[idx]
        if data is None:
            ax.text(0.5, 0.5, "Data Missing", ha='center', va='center')
            continue

        # Create heatmap
        # Rows = Finetuned, Cols = Pretrained
        sns.heatmap(
            data,
            ax=ax,
            cmap="magma",
            vmin=0, vmax=1,
            square=True,
            cbar_kws={"label": measure_title if idx == 1 else None}
        )
        
        # Formatting
        ax.set_title(f"Dataset: {ds_title}", fontsize=16, pad=15)
        ax.set_xlabel("Pretrained Model Layers", fontsize=14)
        if idx == 0:
            ax.set_ylabel("Finetuned Model Layers", fontsize=14)
        
        ax.invert_yaxis()
        
        # Ticks
        num_layers = data.shape[0]
        ticks = range(0, num_layers, 5)
        ax.set_xticks([t + 0.5 for t in ticks])
        ax.set_xticklabels(ticks)
        ax.set_yticks([t + 0.5 for t in ticks])
        ax.set_yticklabels(ticks)

    plt.suptitle(f"{measure_title}: Pretrained vs Finetuned Layer Similarity", fontsize=20, y=0.98)
    plt.tight_layout()
    
    # Save
    save_path = PLOTS_DIR / f"models_similarity_{measure_key}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved plot to {save_path}")
    plt.show()

In [ ]:
# Generate plots for all measures
for key, title in MEASURES.items():
    plot_model_comparison(key, title)